In [ ]:
import random

import geopandas as gpd
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold

### Check the performance of Random Forests with different training samples

In [ ]:
# Set seed for reproducibility
seed = 25
np.random.seed(seed)
random.seed(seed)

# Number of folds in cross-validation
n_folds = 5

# Read the updated SOC dataset (baseline 904 + new 100 SOC observations)
combined_soc_data = r"updated_training_data.gpkg"

# Use optimal hyperparameters found by 5-fold cross-validation (CV) for the baseline random forest (RF) model
best_params = {
    "bootstrap": False,
    "criterion": "squared_error",
    "max_depth": 10,
    "max_features": "sqrt",
    "min_samples_leaf": 4,
    "min_samples_split": 10,
    "n_estimators": 100,
}

# k-fold Nearest Neighbour Distance Matching (kNNDM) folds are made in R (CAST::knndm) by knndm_folds.R and saved as a CSV
knndm_fold_data = r"knndm_5folds.csv"

#### Load the data

In [ ]:
data = gpd.read_file(combined_soc_data)
print("rows and columns:", data.shape)
print("columns:", list(data.columns))

# Which column says whether a sample is old or new?  Print it and check the labels.
source_column = "origin"  # confirmed from the file
print("\nlabels in the source column:")
print(data[source_column].value_counts())

# True for the 100 new samples, False for the 904 legacy samples.
is_new = (
    data[source_column] == "field_work"
).values  # this variable is used to remove any fieldwork samples during the cross validation for baseline RF model
print("\nnew samples found:", is_new.sum(), " legacy samples:", (~is_new).sum())

#### Preprocessing the dataset

In [ ]:
# Separate the dataset into X and y
# Doing get_dummies once (not separately per model) guarantees both models
# see exactly the same feature columns.
X = data.drop(columns=["soc", source_column, "geometry"])
X = pd.get_dummies(X)
y = data["soc"].values

print("\nfeatures used:", X.shape[1])
print("Number of data:", X.shape[0])
print(list(X.columns))

#### Random cross-validation

In [ ]:
# Every sample gets a fold number from 0 to 4.
fold_number = np.zeros(len(data), dtype=int)

kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
for i, (train_rows, test_rows) in enumerate(kf.split(X)):
    fold_number[test_rows] = (
        i  # Assign the fold number to the testing set for each loop
    )

print("\nSamples per fold:", np.bincount(fold_number))
print(
    "New samples per fold:",
    [int(is_new[fold_number == i].sum()) for i in range(n_folds)],
)

In [ ]:
# Create the paired loop
# Two empty arrays to collect one prediction per sample.
pred_baseline = np.full(len(data), np.nan)
pred_updated = np.full(len(data), np.nan)

for i in range(n_folds):
    is_test = fold_number == i  # this fold is the test set
    is_train = ~is_test  # the other 4 folds are the training set

    # Updated RF model: trains on everything in the training folds
    model_updated = RandomForestRegressor(**best_params, random_state=seed)
    model_updated.fit(X[is_train], y[is_train])
    pred_updated[is_test] = model_updated.predict(X[is_test])

    # Baseline model: same training folds, but the new samples are removed
    is_train_baseline = is_train & (~is_new)
    model_baseline = RandomForestRegressor(**best_params, random_state=seed)
    model_baseline.fit(X[is_train_baseline], y[is_train_baseline])
    pred_baseline[is_test] = model_baseline.predict(X[is_test])

    # Full accounting, so you can see where every sample went.
    # The test fold holds both legacy and new samples, which is why the baseline training count is far below 904: the legacy samples sitting in the test fold are held out too.
    legacy_train = int((is_train & ~is_new).sum())
    legacy_test = int((is_test & ~is_new).sum())
    new_train = int((is_train & is_new).sum())
    new_test = int((is_test & is_new).sum())

    print(
        f"fold {i}: test={is_test.sum():4d}  updated_train={is_train.sum():4d}  "
        f"baseline_train={is_train_baseline.sum():4d}  ||  "
        f"legacy {legacy_train}+{legacy_test}={legacy_train + legacy_test}   "
        f"new {new_train}+{new_test}={new_train + new_test}"
    )

    assert legacy_train + legacy_test == 904
    assert new_train + new_test == 100
    assert is_train_baseline.sum() == legacy_train

# Every sample must have received a prediction
assert not np.isnan(pred_baseline).any() and not np.isnan(pred_updated).any()

#### Report performance metrics

In [ ]:
def score(y_true, y_pred):
    """R2, RMSE, MAE and mean error for one set of predictions."""
    return {
        "n": len(y_true),
        "R2": r2_score(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "ME": np.mean(y_pred - y_true),
    }


results = pd.DataFrame(
    [
        score(y, pred_baseline),
        score(y, pred_updated),
    ],
    index=["baseline (904)", "updated (1004)"],
)

print("\nBoth RF models use the same validation set")
print(results.round(2))
print("\nChange after adding the 100 samples:")
print((results.loc["updated (1004)"] - results.loc["baseline (904)"]).round(2))

#### k-fold Nearest Neighbour Distance Matching (kNNDM) cross-validation

kNNDM folds were created beforehand with `knndm_folds.R`: 

- Settings: `knndm(df, modeldomain = baseline_aoa.tif, k = 5)`, seed = 25 (R 4.5.3)
- Method: Linnenbrink et al., 2024, https://doi.org/10.5194/gmd-17-5897-2024
- Software: CAST 1.1.1 (Meyer et al., 2026), https://doi.org/10.32614/CRAN.package.CAST
- The fold file (`knndm_5folds.csv`) has one row per sample in the same row order as `updated_training_data.gpkg`; folds are numbered 0-4.

In [ ]:
knndm = pd.read_csv(knndm_fold_data)

# Check that the fold file has the same rows in the same order as the training data
assert len(knndm) == len(data), "Two files have different lengths"
assert np.allclose(
    knndm[["x", "y"]].values,
    np.c_[data.geometry.x.values, data.geometry.y.values],
    atol=0.01,
), "The rows of the two files are in different orders"

knndm_folds = knndm["fold"].values
print("Samples per fold: ", np.bincount(knndm_folds).tolist())
print(
    "New samples per fold: ",
    [int(is_new[knndm_folds == i].sum()) for i in range(n_folds)],
)

In [ ]:
# Same paired loop as the random CV, but with the kNNDM folds
pred_baseline_knndm = np.full(len(data), np.nan)
pred_updated_knndm = np.full(len(data), np.nan)

for i in range(n_folds):
    is_test = knndm_folds == i
    is_train = ~is_test
    is_train_baseline = is_train & (~is_new)

    model_updated = RandomForestRegressor(**best_params, random_state=seed)
    model_updated.fit(X[is_train], y[is_train])
    pred_updated_knndm[is_test] = model_updated.predict(X[is_test])

    model_baseline = RandomForestRegressor(**best_params, random_state=seed)
    model_baseline.fit(X[is_train_baseline], y[is_train_baseline])
    pred_baseline_knndm[is_test] = model_baseline.predict(X[is_test])

assert (
    not np.isnan(pred_baseline_knndm).any() and not np.isnan(pred_updated_knndm).any()
)

#### Report performance metrics

In [ ]:
knndm_results = pd.DataFrame(
    [score(y, pred_baseline_knndm), score(y, pred_updated_knndm)],
    index=["baseline (904)", "updated (1004)"],
)
print("kNNDM folds")
print(knndm_results.round(2))
print("\nChange after adding the 100 samples:")
print(
    (knndm_results.loc["updated (1004)"] - knndm_results.loc["baseline (904)"]).round(2)
)

# RMSE separately for the original and the new samples
print()
for name, m in [("original 904", ~is_new), ("new 100", is_new)]:
    print(
        f"RMSE {name}: random {score(y[m], pred_baseline[m])['RMSE']:.2f} -> {score(y[m], pred_updated[m])['RMSE']:.2f}"
        f" | kNNDM {score(y[m], pred_baseline_knndm[m])['RMSE']:.2f} -> {score(y[m], pred_updated_knndm[m])['RMSE']:.2f}"
    )

print("\nRandom folds, for comparison")
print(results.round(2))